# Supernova classification workflow demo: ParSNIP and SuperNNova

This notebook runs two pretrained classifiers through [Hyrax](https://hyrax.readthedocs.io) on light curves
stored in a [HATS](https://hats.readthedocs.io) catalog, and validates them on a simulation of Rubin DP2:

1. **Simulate** SN Ia, II and Ib/c light curves on the real Rubin DP2 visits with
   [lightcurvelynx](https://github.com/lincc-frameworks/lightcurvelynx), in the **same schema as the DP2 DIA
   catalog** (`diaObject` rows with nested `diaObjectForcedSource` photometry in nJy), or load a catalog made
   earlier with `simulate_dp2.py`.
2. **ParSNIP** without redshifts (`plasticc_photoz`): predicted redshifts and the latent space.
3. **ParSNIP classification** with a LightGBM classifier trained on **PLAsTiCC** (`parsnip_train_classifier.py`),
   SN Ia vs non-Ia by default.
4. **SuperNNova**: Fink's pretrained ELAsTiCC classifiers (`hyrax_snn`), without and with redshift, and a
   SuperNNova trained on PLAsTiCC if one exists (`snn_train.py`, optional).
5. **Side by side**: SN Ia vs rest for all models, without redshifts.

The simulation is used **only for validation**: nothing in this notebook is trained on it. Because it has the
real-data schema, the same steps run on the DP2 DIA catalog with `examples/classify_rubin_dia.py --method
{parsnip,snn}`, and the diagnostics shown here are the same figures `examples/evaluate.py` makes for any predictions file.

Simulating needs NERSC (the DP2 visit table and lightcurvelynx). Elsewhere, point `DEMO_CATALOG` at a catalog
with the same schema and truth columns. Data go to `data/`, the classifier to `examples/models/` and results
to `examples/results/`, all git-ignored.

**Environment (NERSC).** lightcurvelynx comes from `desc-td-env`, so build the venv on top of it:

```bash
/global/common/software/lsst/install/td_env/2026-09-03-32-23/py/envs/td_env/bin/python \
    -m venv --system-site-packages .venv
.venv/bin/pip install -e ".[notebook]"
.venv/bin/python -m ipykernel install --user --name dp2-sn-classification-td --display-name "dp2-sn-classification (td_env)"
```

and run this notebook with the `dp2-sn-classification (td_env)` kernel.

In [ ]:
import json
import os
from pathlib import Path

import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.table import join
from hyrax import Hyrax
from IPython.display import Image, display

import evaluate  # plot helpers shared with examples/evaluate.py
import hyrax_parsnip
import hyrax_snn

# Paths are relative to the repo root (the notebook runs from examples/), shared with the scripts.
REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
DATA_DIR = REPO / "data"
RESULTS_DIR = REPO / "examples" / "results" / "demo"
FIGURES = RESULTS_DIR / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
# Override with environment variables to run on another catalog or classifier.
CATALOG_PATH = Path(os.environ.get("DEMO_CATALOG", DATA_DIR / "dp2_sim_sne"))
CLASSIFIER_PATH = Path(os.environ.get("DEMO_CLASSIFIER", REPO / "examples" / "models" / "plasticc_classifier.pkl"))
# Optional SuperNNova trained on PLAsTiCC (examples/snn_train.py); used only if it exists.
SNN_PLASTICC_PATH = Path(os.environ.get("DEMO_SNN_MODEL", REPO / "examples" / "models" / "snn_plasticc"))
BANDS = ["u", "g", "r", "i", "z", "y"]


def new_hyrax(batch_size=64):
    h = Hyrax()
    h.set_config("general.results_dir", str(RESULTS_DIR))
    h.set_config("data_loader.batch_size", batch_size)
    return h


def show_evaluation(name, predictions, method, model, redshift_input):
    """examples/evaluate.py's diagnostics for one predictions table (object_id, p_<class>, predicted_class, ...)."""
    table = predictions.assign(
        diaObjectId=predictions["object_id"].astype(np.int64), method=method, model=model, redshift_input=redshift_input
    )
    table, classes = evaluate.prepare(table, truth)
    result = evaluate.evaluate(truth_frame, table, classes, FIGURES / name)
    for figure in result["figures"]:
        display(Image(figure))
    return result["metrics"]

## 1. DP2 light curves as a HATS catalog

Both classifiers read a **nested** HATS catalog with one row per object. We write the simulations with the
column names of the DP2 DIA catalog, plus truth columns:

| column                                   | type                                               |
|------------------------------------------|----------------------------------------------------|
| `diaObjectId`, `ra`, `dec`, `nDiaSources` | per object, as in DP2                             |
| `diaObjectForcedSource`                  | nested: `midpointMjdTai`, `band`, `psfDiffFlux`, `psfDiffFluxErr` (nJy), `visit`, flags |
| `type`, `redshift`, `t0`, `mwebv`, `template` | truth (simulations only)                      |

`hyrax_parsnip.RUBIN_DIA_SETTINGS` maps these columns for the dataset, rescales nJy to the PLAsTiCC zeropoint
(27.5), and drops flagged points.

The simulation forward-models each supernova through the **DP2 visit-detector table**: every CCD visit
covering its position gives one forced-photometry point, with noise from that visit's sky, zeropoint and seeing
(Poisson errors inflated by 1.5, as in `dp2_sims_visitdetector`). Positions are drawn uniformly over the
area covered by both DP2 visits and the DP2 DIA catalog, and Milky Way extinction comes from SFD.

- **SN Ia**: SALT3, redshift from the Frohmaier et al. (2019) volumetric rate, asymmetric Gaussian priors on
  `x1` and `c`, and `x0` from the Tripp relation (same priors as `dp2_sims_visitdetector`).
- **SN II** (II-P/L, IIn, IIb) and **SN Ib/c** (Ib, Ic, Ic-BL): one of the Vincenzi et al. (2019) templates
  per object, scaled to a Gaussian peak absolute B magnitude, with a rate ∝ (1+z)^2.7.

A DIA object only exists if the source was detected, so `simulate_catalog` keeps objects with at least 2
unsaturated points above SNR 5, and simulates in batches until each class has `N_PER_CLASS`.
The first run downloads the templates into the sncosmo cache.

If `CATALOG_PATH` already exists (e.g. from `python examples/simulate_dp2.py data/dp2_sim_sne`), it is loaded
and the simulation cells below are skipped.

In [ ]:
SEED = 1024
N_PER_CLASS = 1000  # detected objects kept per class
SIMULATE = not CATALOG_PATH.exists()

try:
    import simulate_dp2  # needs lightcurvelynx (desc-td-env at NERSC)
except ImportError:
    simulate_dp2 = None
    if SIMULATE:
        raise

print(f"{'Simulating' if SIMULATE else 'Loading'} {CATALOG_PATH}")

Load the DP2 visit-detector table (about 5 million CCD visits) and the sky area to simulate: (only when simulating)

In [ ]:
if SIMULATE:
    survey = simulate_dp2.load_survey(noise_scale=1.5)  # DP2 flux errors are underestimated
    print(f"{len(survey.survey_info.obstable):,} CCD visits, MJD {survey.t_min:.1f}-{survey.t_max:.1f}")
    print(f"Simulated area: {survey.footprint.sky_fraction * 41253:.0f} deg²")

Simulate each class and write the HATS catalog. lightcurvelynx fluxes are already in nJy, like Rubin's.
Saturated points get `psfDiffFlux_flag`; the other flags the dataset checks are all False.

In [ ]:
if SIMULATE:
    simulated = simulate_dp2.simulate_catalog(survey, N_PER_CLASS, seed=SEED)
    simulate_dp2.write_catalog(simulated, CATALOG_PATH)

Check that the simulated catalog has the DP2 DIA schema: every column it shares with the real catalog has the same
type, and the only extra columns are the truth columns. This needs the real DP2 DIA catalog, so it only runs at NERSC.

In [ ]:
if simulate_dp2 is not None and Path(simulate_dp2.DP2_DIA_CATALOG).exists():
    print("simulation-only columns:", simulate_dp2.check_schema(CATALOG_PATH))
else:
    print("Skipped: the real DP2 DIA catalog is not available here.")

Open the catalog with LSDB to check what it looks like:

In [ ]:
catalog = lsdb.open_catalog(CATALOG_PATH)
frame = catalog.compute()
CLASSES = sorted(frame["type"].unique())
truth_frame, truth = evaluate.load_truth(CATALOG_PATH)  # truth + light-curve S/N, for the diagnostics
print(f"{len(frame)} objects, columns: {list(frame.columns)}")
print(frame["type"].value_counts().to_dict())
shown = [c for c in ["diaObjectId", "type", "template", "redshift", "nDiaSources", "diaObjectForcedSource"] if c in frame]
frame[shown].head()

In [ ]:
BAND_COLORS = dict(zip(BANDS, ["tab:purple", "tab:blue", "tab:green", "tab:orange", "tab:red", "tab:brown"]))


def plot_lightcurve(ax, row, title=None):
    # Forced photometry covers all of DP2, so show -30 to +100 days around the true t0.
    lc = row["diaObjectForcedSource"]
    lc = lc[(lc["midpointMjdTai"] > row["t0"] - 30) & (lc["midpointMjdTai"] < row["t0"] + 100)]
    for band in BANDS:
        sel = lc[lc["band"] == band]
        ax.errorbar(
            sel["midpointMjdTai"], sel["psfDiffFlux"], sel["psfDiffFluxErr"], fmt="o", ms=3, color=BAND_COLORS[band], label=band
        )
    ax.set_title(title or f"{row['diaObjectId']} ({row['type']}, z={row['redshift']:.2f})")
    ax.set_xlabel("MJD")


# The best-sampled object of each class.
examples = pd.concat([frame[frame["type"] == c].sort_values("nDiaSources").tail(1) for c in CLASSES])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (_, row) in zip(axes, examples.iterrows()):
    plot_lightcurve(ax, row)
axes[0].set_ylabel("psfDiffFlux (nJy)")
axes[-1].legend(ncol=2, fontsize=8)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label in CLASSES:
    sel = frame[frame["type"] == label]
    axes[0].hist(sel["redshift"], bins=30, histtype="step", label=label)
    axes[1].hist(sel["nDiaSources"], bins=np.arange(0, 60, 2), histtype="step", label=label)
axes[0].set_xlabel("redshift")
axes[1].set_xlabel("detections (nDiaSources)")
axes[0].legend()
plt.tight_layout()

## 2. ParSNIP without redshifts

`hyrax_parsnip.configure` points a Hyrax instance at the catalog and a pretrained ParSNIP model, so `h.infer()`
runs without a training run. Real DIA objects have no spectroscopic redshift, so we use `plasticc_photoz`, which
predicts the redshift from the light curve (with a weak, constant host photo-z prior). `RUBIN_DIA_SETTINGS`
points the dataset at the DIA columns, rescales nJy to the PLAsTiCC zeropoint (27.5), drops flagged points and
uses no redshift column. The simulated Milky Way E(B-V) is used for the extinction correction.

`load_predictions` returns one row per object (Hyrax keeps ids as strings, so `object_id` is the `diaObjectId`
as a string). Objects ParSNIP can't process get NaN features instead of being dropped.

(`plasticc`, which takes the true redshift as input, also works here, but a classifier trained on its features
would be needed, and the result would include redshift information.)

The predicted redshifts and the latent space are among the diagnostics of section 3.

In [ ]:
h = new_hyrax()
hyrax_parsnip.configure(h, CATALOG_PATH, pretrained="plasticc_photoz")
for key, value in hyrax_parsnip.RUBIN_DIA_SETTINGS.items():
    h.set_config(f"data_set.ParsnipHATSDataset.{key}", value)
h.set_config("data_set.ParsnipHATSDataset.mwebv_column", "mwebv")

labels = hyrax_parsnip.catalog_metadata(CATALOG_PATH, ["type", "redshift"], id_column="diaObjectId")
parsnip_features = hyrax_parsnip.load_predictions(h.infer(), metadata=labels)
print(f"{len(parsnip_features)} objects, {len(parsnip_features.colnames)} columns")
parsnip_features["object_id", "type", "redshift", "predicted_redshift", "s1", "s2", "s3", "color", "luminosity"][:5]

## 3. ParSNIP classification with a PLAsTiCC classifier

ParSNIP doesn't ship a classifier. Since this simulation is for validation, the LightGBM classifier is trained
on **PLAsTiCC** instead, on the same `plasticc_photoz` features. By default it is binary, **SNIa vs non-SNIa**
(`--classes dp2` gives SNIa, SNII, SNIbc and "other").
If `CLASSIFIER_PATH` doesn't exist yet, the next cell makes it with the same two scripts as the command line:

```bash
python examples/plasticc_to_hats.py data/plasticc_hats --ddf
python examples/parsnip_train_classifier.py data/plasticc_hats   # writes examples/models/plasticc_classifier.pkl
```

By default the training sample is the official training set plus the test-set DDF objects (40,774 objects;
~330 MB download, roughly 35 minutes of ParSNIP on one CPU). Set `DEMO_PLASTICC_DDF=0` for the training set only
(7,848 objects, ~7 minutes), e.g. for a quick check. At NERSC, `nersc/train_parsnip_classifier.sbatch` does the
same in a batch job. Later runs reuse the classifier, whatever its classes (its `.json` is printed below).

The classifier's `.json` records how its features were made; it must match the model used above.

In [ ]:
import re
import subprocess
import sys

PLASTICC_DDF = os.environ.get("DEMO_PLASTICC_DDF", "1") != "0"
PLASTICC_CATALOG = DATA_DIR / ("plasticc_hats" if PLASTICC_DDF else "plasticc_hats_train")


# Skip progress bars and library logging in the scripts' output; keep their own lines (counts, K-fold results).
QUIET = re.compile("|".join([
    r"it/s\]", r"s/it\]",  # progress bars
    r"^\[20\d\d-", r" INFO: ", r"^\[LightGBM\]", r"force_(col|row)_wise",  # Hyrax, ignite, LightGBM logging
    r"^Training classifier with keys", r"^ {4}[a-z_0-9]+\s*$",  # ParSNIP's list of classifier features
    r"^\s*(\{'|\}|>|Name:|Dataset class|Data location|Selected items|Primary ID|Requested fields|'sampler')",  # Hyrax dumps
    r"^\s*$",
]))


def run_script(*args):
    """Run an example script from the repo root, streaming its output into the notebook."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONWARNINGS": "ignore", "MLFLOW_DISABLE_AGENT_HINT": "1"}
    with subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            if not QUIET.search(line):
                print(line, end="")
    if proc.returncode:
        raise RuntimeError(f"{args[0]} failed (exit code {proc.returncode})")


if CLASSIFIER_PATH.exists():
    print(f"Using the existing classifier {CLASSIFIER_PATH}")
else:
    if not PLASTICC_CATALOG.exists():
        run_script(
            "examples/plasticc_to_hats.py", PLASTICC_CATALOG, "--raw-dir", DATA_DIR / "plasticc_raw",
            *(["--ddf"] if PLASTICC_DDF else []),
        )
    run_script(
        "examples/parsnip_train_classifier.py", PLASTICC_CATALOG,
        "--output", CLASSIFIER_PATH, "--results-dir", RESULTS_DIR / "train_classifier",
    )

In [ ]:
info_path = CLASSIFIER_PATH.with_suffix(".json")
if info_path.exists():
    info = json.loads(info_path.read_text())
    assert info["model"] == "plasticc_photoz" and not info["redshift_input"], (
        f"{CLASSIFIER_PATH} was trained on {info['model']} features (redshift input: {info['redshift_input']})"
    )
    print(f"Classifier ({info['class_scheme']}): trained on {info['counts']} ({info['samples']}), "
          f"K-fold accuracy {info['kfold_accuracy']:.3f}")
classifier = hyrax_parsnip.load_classifier(CLASSIFIER_PATH)
probabilities = hyrax_parsnip.classify(classifier, parsnip_features)
parsnip_classes = probabilities.colnames[1:]

parsnip_table = join(probabilities, parsnip_features, keys="object_id").to_pandas().drop(columns="original_object_id")
parsnip_table = parsnip_table.rename(columns={c: f"p_{c}" for c in parsnip_classes})
p = parsnip_table[[f"p_{c}" for c in parsnip_classes]].to_numpy()
parsnip_table["predicted_class"] = np.where(
    np.isfinite(p).all(axis=1), np.asarray(parsnip_classes)[np.nan_to_num(p, nan=-1).argmax(axis=1)], ""
)
print(parsnip_table["predicted_class"].value_counts().to_dict())

The same diagnostics as `examples/evaluate.py` (scored against the simulation truth; nothing is trained on it):

In [ ]:
parsnip_metrics = show_evaluation("parsnip", parsnip_table, "parsnip", "plasticc_photoz", redshift_input=False)
parsnip_metrics

## 4. SuperNNova

`hyrax_snn` runs Fink's pretrained SuperNNova classifiers (trained on ELAsTiCC; Fraga et al. 2024) on the same
catalog, with no training. They have fixed classes: `elasticc_ia` is SN Ia vs other without redshift;
`elasticc_broad` predicts SN, Fast, Long, Periodic and NonPeriodic and needs a redshift. `RUBIN_DIA_SETTINGS`
keeps the fluxes in nJy, which these models behave as if trained on.

How the input is prepared (`model.HyraxSNN.time_window`, `detection_snr`, flux units; `FINK_EXACT` approximates
Fink's own processing) still has to be chosen on this simulation: the defaults are a starting point.

In [ ]:
from hyrax_snn.pretrained import resolve


def run_snn(model, **dataset_settings):
    """A built-in model name or a model directory; the flux scale is the one the model expects."""
    h = new_hyrax()
    settings = {**hyrax_snn.rubin_dia_settings(model), "mwebv_column": "mwebv", **dataset_settings}
    hyrax_snn.configure(h, CATALOG_PATH, pretrained=model, dataset_settings=settings)
    table = hyrax_snn.load_predictions(h.infer(), metadata=labels).to_pandas()
    classes = resolve(model).class_names
    return table.rename(columns={c: f"p_{c}" for c in classes}), classes


snn_ia, _ = run_snn("elasticc_ia")
snn_ia["predicted_class"].value_counts().to_dict()

The same diagnostics for `elasticc_ia` (SN Ia vs other, no redshift):

In [ ]:
snn_ia_metrics = show_evaluation("snn_ia", snn_ia, "snn", "elasticc_ia", redshift_input=False)
snn_ia_metrics

`elasticc_broad` needs a redshift, so here it is given the **true** redshift (not available for real DIA
objects; the figures are marked). Every simulated object is a supernova, so ideally all would be classified "SN":

In [ ]:
snn_broad, broad_classes = run_snn("elasticc_broad", redshift_column="redshift")
snn_broad_metrics = show_evaluation("snn_broad", snn_broad, "snn", "elasticc_broad", redshift_input=True)
snn_broad_metrics

### SuperNNova trained on PLAsTiCC (optional)

There are no pretrained SuperNNova models for PLAsTiCC, so to compare with ParSNIP on the **same training
data**, SuperNNova can be trained on PLAsTiCC with Hyrax (SN Ia vs non-Ia, no redshift, SuperNNova's default
network and training settings):

```bash
python examples/snn_train.py data/plasticc_hats   # writes examples/models/snn_plasticc/
```

(About an hour on one CPU for the training set + DDF; `nersc/snn_train.sbatch` at NERSC.) This notebook doesn't
train it: the cells below use `SNN_PLASTICC_PATH` only if it exists. The model records the zeropoint of its
training fluxes (27.5 for PLAsTiCC), so `rubin_dia_settings` rescales the nJy fluxes to match.

In [ ]:
snn_plasticc = None
if (SNN_PLASTICC_PATH / "model.pt").exists():
    training = json.loads((SNN_PLASTICC_PATH / "cli_args.json").read_text()).get("training", {})
    print(f"Using {SNN_PLASTICC_PATH}: trained on {training.get('catalog')} ({training.get('class_scheme')}), "
          f"validation accuracy {training.get('validation_accuracy', float('nan')):.3f}")
    snn_plasticc, _ = run_snn(str(SNN_PLASTICC_PATH))
    snn_plasticc_metrics = show_evaluation("snn_plasticc", snn_plasticc, "snn", "snn_plasticc", redshift_input=False)
    display(snn_plasticc_metrics)
else:
    print(f"No SuperNNova trained on PLAsTiCC at {SNN_PLASTICC_PATH}; skipped (see the command above).")

## 5. Side by side: SN Ia vs rest, without redshifts

Both models on the same objects, none given a redshift: ParSNIP (`plasticc_photoz` features + PLAsTiCC
classifier), SuperNNova `elasticc_ia`, and SuperNNova trained on PLAsTiCC if it was run above.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

scores = {
    "SuperNNova elasticc_ia": snn_ia[["object_id", "type", "p_SNIa"]],
    "ParSNIP + PLAsTiCC classifier": parsnip_table[["object_id", "type", "p_SNIa"]],
}
if snn_plasticc is not None:
    scores["SuperNNova trained on PLAsTiCC"] = snn_plasticc[["object_id", "type", "p_SNIa"]]
common = set.intersection(*(set(s["object_id"]) for s in scores.values()))

fig, ax = plt.subplots(figsize=(4.8, 4.2))
rows = []
for name, s in scores.items():
    s = s[s["object_id"].isin(common) & np.isfinite(s["p_SNIa"])]
    is_ia = (s["type"] == "SNIa").to_numpy()
    selected = (s["p_SNIa"] >= 0.5).to_numpy()
    fpr, tpr, _ = roc_curve(is_ia, s["p_SNIa"])
    auc = roc_auc_score(is_ia, s["p_SNIa"])
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC {auc:.3f})")
    rows.append({
        "model": name, "objects": len(s), "AUC": auc,
        "efficiency@0.5": selected[is_ia].mean(),
        "purity@0.5": is_ia[selected].mean() if selected.any() else np.nan,
    })
ax.plot([0, 1], [0, 1], color="#999999", ls=":", lw=1)
ax.set(xlabel="false positive rate", ylabel="true positive rate", title="SN Ia vs rest, no redshift")
ax.legend(fontsize=8)
plt.tight_layout()
pd.DataFrame(rows).round(3)

## Next steps

- The same steps as scripts, for larger runs and the real data (the simulated catalog has the same schema):
  ```bash
  python examples/classify_rubin_dia.py data/dp2_sim_sne --method parsnip --classifier examples/models/plasticc_classifier.pkl
  python examples/classify_rubin_dia.py data/dp2_sim_sne --method snn
  python examples/evaluate.py data/dp2_sim_sne examples/results/parsnip_predictions.parquet
  python examples/evaluate.py data/dp2_sim_sne examples/results/snn_predictions.parquet
  ```
  Then run `classify_rubin_dia.py` on the DP2 DIA catalog (its default path).
- Choose SuperNNova's input options on this simulation (`--detection-snr`, `--no-time-window`, `--fink-exact`).
- Train SuperNNova on PLAsTiCC (`examples/snn_train.py`) and classify with
  `classify_rubin_dia.py --method snn --model examples/models/snn_plasticc`.
- Fine-tuning ParSNIP itself: `examples/parsnip_train_then_infer.py`.
- See `README.md` for macOS notes (CPU device, single-threaded LightGBM) and known limitations.